# Outils d'analyse des données trafic

## Nettoyage et agrégation par flag

In [2]:
import polars as pl
from datetime import datetime

def process_siredo_traffic(df: pl.DataFrame) -> pl.DataFrame:
    """
    Traite un DataFrame de données individuelles SIREDO (spm_bs_individuelle) :
    1. Ajout de flags de qualité (présence vs vitesse valide).
    2. Agrégation par station et par pas de temps (6 minutes).
    3. Calcul propre des débits et des vitesses moyennes.
    """
    
    # Étape 1 : Application des flags de qualification
    df_flagged = df.with_columns([
        # Flag 1 : Présence physique valide (exclut les erreurs fatales de capteurs : 
        # vitesses aberrantes hors limites physiques absolues ou longueurs négatives/nulles)
        (
            (pl.col("vitesse") >= 0) & 
            (pl.col("vitesse") <= 200) & 
            (pl.col("longueur") > 0)
        ).alias("is_valid_presence"),
        
        # Flag 2 : Vitesse exploitable pour les analyses statistiques 
        # (exclut les excès extrêmes ou les artefacts de couplage de boucles)
        (
            (pl.col("vitesse") >= 10) & 
            (pl.col("vitesse") <= 160)
        ).alias("is_valid_speed")
    ])
    
    # Étape 2 : Troncature temporelle et agrégation
    df_aggregated = (
        df_flagged
        .with_columns(
            # Découpage temporel standard SIREDO (ex: pas de 6 minutes)
            intervalle = pl.col("timestamp").dt.truncate("6m")
        )
        .group_by(["id_station", "intervalle"])
        .agg([
            # Débit brut (tous les enregistrements reçus)
            pl.len().alias("debit_brut"),
            
            # Débit réel/valide : on compte tous les véhicules ayant physiquement franchi 
            # la station (on ne supprime pas le véhicule du débit sous prétexte qu'il roule vite)
            pl.col("is_valid_presence").sum().alias("debit_valide"),
            
            # Vitesse moyenne calculée uniquement sur les véhicules dont la vitesse est valide
            pl.when(pl.col("is_valid_speed"))
              .then(pl.col("vitesse"))
              .mean()
              .alias("vitesse_moyenne"),
              
            # Indicateur de qualité : taux de rejet des vitesses sur l'intervalle
            (
                1.0 - (pl.col("is_valid_speed").sum() / pl.col("is_valid_presence").sum().clip(lower_bound=1))
            ).alias("taux_rejet_vitesse")
        ])
        .sort(["id_station", "intervalle"])
    )
    
    return df_aggregated


# ==========================================
# Exemple d'utilisation avec des données simulées
# ==========================================
if __name__ == "__main__":
    # Simulation d'un petit jeu de données SIREDO individuel
    data = {
        "id_station": ["STA_01", "STA_01", "STA_01", "STA_01", "STA_01"],
       "timestamp": [
            datetime(2026, 10, 14, 8, 0, 2),
            datetime(2026, 10, 14, 8, 0, 15),
            datetime(2026, 10, 14, 8, 2, 10),
            datetime(2026, 10, 14, 8, 5, 45),
            datetime(2026, 10, 14, 8, 6, 12),
        ],
        "vitesse": [135.0, 145.0, -1.0, 185.0, 90.0],  # -1 (erreur capteur), 185 (aberrant/technique)
        "longueur": [4.5, 16.2, 4.0, 5.0, 4.2],         # Véhicule 2 est un Poids Lourd
        "classe": ["VL", "PL", "VL", "VL", "VL"]
    }
    
    df_brut = pl.DataFrame(data)
    
    print("--- Données Brutes ---")
    print(df_brut)
    
    df_resultat = process_siredo_traffic(df_brut)
    
    print("\n--- Données Agrégées et Nettoyées (Pas de 6 min) ---")
    print(df_resultat)

--- Données Brutes ---
shape: (5, 5)
┌────────────┬─────────────────────┬─────────┬──────────┬────────┐
│ id_station ┆ timestamp           ┆ vitesse ┆ longueur ┆ classe │
│ ---        ┆ ---                 ┆ ---     ┆ ---      ┆ ---    │
│ str        ┆ datetime[μs]        ┆ f64     ┆ f64      ┆ str    │
╞════════════╪═════════════════════╪═════════╪══════════╪════════╡
│ STA_01     ┆ 2026-10-14 08:00:02 ┆ 135.0   ┆ 4.5      ┆ VL     │
│ STA_01     ┆ 2026-10-14 08:00:15 ┆ 145.0   ┆ 16.2     ┆ PL     │
│ STA_01     ┆ 2026-10-14 08:02:10 ┆ -1.0    ┆ 4.0      ┆ VL     │
│ STA_01     ┆ 2026-10-14 08:05:45 ┆ 185.0   ┆ 5.0      ┆ VL     │
│ STA_01     ┆ 2026-10-14 08:06:12 ┆ 90.0    ┆ 4.2      ┆ VL     │
└────────────┴─────────────────────┴─────────┴──────────┴────────┘

--- Données Agrégées et Nettoyées (Pas de 6 min) ---
shape: (2, 6)
┌────────────┬──────────────┬────────────┬──────────────┬─────────────────┬────────────────────┐
│ id_station ┆ intervalle   ┆ debit_brut ┆ debit_valide ┆ vit

In [ ]:
import polars as pl
import numpy as np
from scipy.optimize import curve_fit

# ==========================================
# 1. Définition des modèles mathématiques (Vitesse = f(Densité))
# ==========================================
def model_exponentiel(rho, v_f, rho_0):
    """Modèle exponentiel d'Underwood"""
    return v_f * np.exp(-rho / rho_0)

def model_puissance(rho, v_f, rho_j, n):
    """Modèle de puissance (type Drake généralisé)"""
    # On s'assure de ne pas élever un nombre négatif à une puissance non entière
    term = np.maximum(0.0, 1.0 - (rho / rho_j) ** n)
    return v_f * term


def caler_diagramme_fondamental(df_agg: pl.DataFrame) -> dict:
    """
    Calcule la densité et ajuste les modèles exponentiel et puissance.
    """
    # Conversion en Pandas pour l'interfaçage fluide avec SciPy/NumPy
    df_pd = df_agg.drop_nulls(subset=["debit_valide", "vitesse_moyenne"]).to_pandas()
    
    # Conversion du débit 6 minutes en débit horaire (Q_h = Q_6min * 10)
    # Densité en veh/km = Débit horaire (veh/h) / Vitesse moyenne (km/h)
    df_pd["debit_horaire"] = df_pd["debit_valide"] * 10.0
    
    # Filtrage des vitesses nulles ou aberrantes pour éviter les divisions par zéro
    mask = (df_pd["vitesse_moyenne"] > 10.0) & (df_pd["debit_horaire"] > 50.0)
    df_clean = df_pd[mask].copy()
    
    rho = df_clean["debit_horaire"].values / df_clean["vitesse_moyenne"].values
    v = df_clean["vitesse_moyenne"].values
    
    # ------------------------------------------
    # Ajustement du Modèle Exponentiel
    # ------------------------------------------
    try:
        # Estimation initiale : v_f ~ max(v), rho_0 ~ 50
        popt_exp, _ = curve_fit(model_exponentiel, rho, v, p0=[130.0, 40.0], bounds=([50.0, 5.0], [160.0, 200.0]))
        v_f_exp, rho_0_exp = popt_exp
        v_pred_exp = model_exponentiel(rho, *popt_exp)
        
        # Calcul du R2
        ss_res_exp = np.sum((v - v_pred_exp) ** 2)
        ss_tot = np.sum((v - np.mean(v)) ** 2)
        r2_exp = 1 - (ss_res_exp / ss_tot)
    except Exception as e:
        popt_exp, r2_exp = None, np.nan
        print(f"Erreur fit exponentiel : {e}")

    # ------------------------------------------
    # Ajustement du Modèle de Puissance
    # ------------------------------------------
    try:
        # Estimation initiale : v_f ~ max(v), rho_j ~ 120 veh/km, n ~ 1.0
        popt_pow, _ = curve_fit(model_puissance, rho, v, p0=[130.0, 120.0, 1.0], bounds=([50.0, 60.0, 0.1], [160.0, 250.0, 5.0]))
        v_f_pow, rho_j_pow, n_pow = popt_pow
        v_pred_pow = model_puissance(rho, *popt_pow)
        
        # Calcul du R2
        ss_res_pow = np.sum((v - v_pred_pow) ** 2)
        r2_pow = 1 - (ss_res_pow / ss_tot)
    except Exception as e:
        popt_pow, r2_pow = None, np.nan
        print(f"Erreur fit puissance : {e}")

    return {
        "rho": rho,
        "v_observee": v,
        "exp_params": {"v_f": v_f_exp, "rho_0": rho_0_exp, "r2": r2_exp},
        "pow_params": {"v_f": v_f_pow, "rho_j": rho_j_pow, "n": n_pow, "r2": r2_pow}
    }

## Adaptation du code ci dessus pour le modele de puissance généralisé

In [ ]:
import polars as pl
import numpy as np
from scipy.optimize import curve_fit

def model_puissance_custom(rho, a, b, alpha):
    """
    Modèle de puissance personnalisé : V = a + b * K^alpha
    Où a > 0 (vitesse à flux libre) et b < 0 (pente de décélération).
    """
    # On protège l'exposant pour éviter les erreurs numériques si rho est proche de 0
    return a + b * (np.maximum(0.0, rho) ** alpha)


def caler_diagramme_puissance_custom(df_agg: pl.DataFrame) -> dict:
    """
    Ajuste le modèle V = a + b * K^alpha sur les données SIREDO agrégées.
    """
    df_pd = df_agg.drop_nulls(subset=["debit_valide", "vitesse_moyenne"]).to_pandas()
    
    # Conversion en débit horaire et calcul de la densité K (veh/km)
    df_pd["debit_horaire"] = df_pd["debit_valide"] * 10.0
    
    # Filtrage pour éviter les aberrations et divisions par zéro
    mask = (df_pd["vitesse_moyenne"] > 10.0) & (df_pd["debit_horaire"] > 50.0)
    df_clean = df_pd[mask].copy()
    
    k = df_clean["debit_horaire"].values / df_clean["vitesse_moyenne"].values
    v = df_clean["vitesse_moyenne"].values
    
    # Ajustement par moindres carrés non linéaires
    try:
        # Estimations initiales : a ~ 130, b ~ -0.5, alpha ~ 1.0
        # Bornes : a entre 90 et 150, b négatif strict, alpha entre 0.1 et 3.0
        popt, _ = curve_fit(
            model_puissance_custom, 
            k, 
            v, 
            p0=[130.0, -0.5, 1.0], 
            bounds=([90.0, -100.0, 0.1], [150.0, 0.0, 3.0])
        )
        a_opt, b_opt, alpha_opt = popt
        v_pred = model_puissance_custom(k, *popt)
        
        # Calcul du coefficient de détermination R2
        ss_res = np.sum((v - v_pred) ** 2)
        ss_tot = np.sum((v - np.mean(v)) ** 2)
        r2 = 1 - (ss_res / ss_tot)
        
    except Exception as e:
        popt, r2 = None, np.nan
        print(f"Erreur lors du fit de la puissance : {e}")
        return {}

    return {
        "k": k,
        "v_observee": v,
        "params": {"a": a_opt, "b": b_opt, "alpha": alpha_opt},
        "r2": r2
    }

## Modelisation de l'effet PL

In [ ]:
import polars as pl
import numpy as np
from scipy.optimize import curve_fit

def model_puissance_mixte(X, a, b, alpha, Cvp):
    """
    Modèle de puissance pour trafic mixte :
    X[0] = K (densité en veh/km)
    X[1] = p (proportion de PL, entre 0 et 1)
    """
    k, p = X
    num = a + b * (np.maximum(0.0, k) ** alpha)
    den = 1.0 + (Cvp - 1.0) * p
    return num / den


def caler_diagramme_avec_pl(df_agg: pl.DataFrame) -> dict:
    """
    Ajuste le modèle V = (a + b * K^alpha) / (1 + (Cvp - 1) * p)
    à partir des données SIREDO agrégées contenant les débits VL et PL.
    """
    df_pd = df_agg.drop_nulls(subset=["debit_vl", "debit_pl", "vitesse_moyenne"]).to_pandas()
    
    # 1. Calcul des débits horaires (Q_6min * 10)
    df_pd["debit_vl_h"] = df_pd["debit_vl"] * 10.0
    df_pd["debit_pl_h"] = df_pd["debit_pl"] * 10.0
    df_pd["debit_total_h"] = df_pd["debit_vl_h"] + df_pd["debit_pl_h"]
    
    # 2. Proportion de PL (p)
    df_pd["p"] = df_pd["debit_pl_h"] / df_pd["debit_total_h"].clip(lower_bound=1.0)
    
    # 3. Densité K (veh/km) basée sur la vitesse moyenne observée
    df_pd["k"] = df_pd["debit_total_h"] / df_pd["vitesse_moyenne"]
    
    # Filtrage de cohérence
    mask = (df_pd["vitesse_moyenne"] > 10.0) & (df_pd["debit_total_h"] > 50.0)
    df_clean = df_pd[mask].copy()
    
    k = df_clean["k"].values
    p = df_clean["p"].values
    v = df_clean["vitesse_moyenne"].values
    
    try:
        # Estimation initiale : a ~ 130, b ~ -0.5, alpha ~ 1.0, Cvp ~ 2.0
        # Bornes pour contraindre les paramètres physiquement
        popt, _ = curve_fit(
            model_puissance_mixte, 
            xdata=(k, p), 
            ydata=v, 
            p0=[130.0, -0.5, 1.0, 2.0],
            bounds=([90.0, -100.0, 0.1, 1.0], [150.0, 0.0, 3.0, 5.0])
        )
        a_opt, b_opt, alpha_opt, cvp_opt = popt
        v_pred = model_puissance_mixte((k, p), *popt)
        
        # Calcul du R2
        ss_res = np.sum((v - v_pred) ** 2)
        ss_tot = np.sum((v - np.mean(v)) ** 2)
        r2 = 1 - (ss_res / ss_tot)
        
    except Exception as e:
        popt, r2 = None, np.nan
        print(f"Erreur lors du fit mixte : {e}")
        return {}

    return {
        "params": {"a": a_opt, "b": b_opt, "alpha": alpha_opt, "Cvp": cvp_opt},
        "r2": r2
    }